## Team STAR Part 1: Get comfortable using Tsim
![surface_code.png](./assets/surface_code.png)

Familiarize yourself with the surface code and its stabilizer structure. 
Use Tsim to construct a distance-3 surface code and simulate two rounds of syndrome extraction.

Start by identifying the data and ancilla qubits, the stabilizer checks being measured, and how syndrome information is extracted over time. 
Use this part to get comfortable building the code in Tsim, running repeated syndrome cycles, and understanding how errors affect the circuit.

**Goal:** Build intuition for the distance-3 surface code, repeated syndrome extraction, and the simulation workflow in Tsim.


In [4]:
# Using Squin
from typing import Any

from bloqade import squin, tsim
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
# import tsim

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)


@squin.kernel
def bell_state() -> Measurement:
    qubits = squin.qalloc(2)
    squin.h(qubits[0])
    squin.cx(qubits[0], qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

show_circuit(bell_state)


In [ ]:
# SQUIN surface code
@squin.kernel
def distance_3_surface() -> Measurement:
    # qubits 0-8: data, 9-12: Z-ancillas, 13-16: X-ancillas
    qubits = squin.qalloc(17)

    # Z-ancillas → |0⟩, X-ancillas → |+⟩
    for i in range(9, 17):
        squin.reset(qubits[i])
    for i in range(13, 17):
        squin.h(qubits[i])

    # CNOT step 1 — X: 13→0, 14→2, 16→4  |  Z: 1→10, 3→11, 7→12
    squin.cx(qubits[13], qubits[0])
    squin.cx(qubits[14], qubits[2])
    squin.cx(qubits[16], qubits[4])
    squin.cx(qubits[1],  qubits[10])
    squin.cx(qubits[3],  qubits[11])
    squin.cx(qubits[7],  qubits[12])

    # CNOT step 2 — X: 13→3, 14→5, 16→7  |  Z: 2→10, 4→11, 8→12
    squin.cx(qubits[13], qubits[3])
    squin.cx(qubits[14], qubits[5])
    squin.cx(qubits[16], qubits[7])
    squin.cx(qubits[2],  qubits[10])
    squin.cx(qubits[4],  qubits[11])
    squin.cx(qubits[8],  qubits[12])

    # CNOT step 3 — X: 13→1, 15→3, 16→5  |  Z: 0→9, 4→10, 6→11
    squin.cx(qubits[13], qubits[1])
    squin.cx(qubits[15], qubits[3])
    squin.cx(qubits[16], qubits[5])
    squin.cx(qubits[0],  qubits[9])
    squin.cx(qubits[4],  qubits[10])
    squin.cx(qubits[6],  qubits[11])

    # CNOT step 4 — X: 13→4, 15→6, 16→8  |  Z: 1→9, 5→10, 7→11
    squin.cx(qubits[13], qubits[4])
    squin.cx(qubits[15], qubits[6])
    squin.cx(qubits[16], qubits[8])
    squin.cx(qubits[1],  qubits[9])
    squin.cx(qubits[5],  qubits[10])
    squin.cx(qubits[7],  qubits[11])

    # X-ancillas: rotate to Z-basis before measurement
    for i in range(13, 17):
        squin.h(qubits[i])

    # syndromes[0:4] = Z-ancillas (q9-q12), syndromes[4:8] = X-ancillas (q13-q16)
    syndromes = squin.broadcast.measure(
        [qubits[9],  qubits[10], qubits[11], qubits[12],
         qubits[13], qubits[14], qubits[15], qubits[16]]
    )
    return syndromes

show_circuit(distance_3_surface)

NotImplementedError: Missing implementation for New at File "/home/ruzza/RepoRoot/ETH2_test/ETH_Qhack_26_Quantum_Icing/2026/.venv/lib/python3.12/site-packages/bloqade/qubit/stdlib/_new.py", line 32, col 15

In [5]:
####

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tsim

# ─────────────────────────────────────────────────────────────────────────────
# 1. Distance-3 Rotated Surface Code: qubit layout & stabilizers
# ─────────────────────────────────────────────────────────────────────────────
# Qubit indices follow the convention in the STAR stim circuits:
#   Data qubits   : 0–8  (3×3 grid, half-integer coordinates)
#   Z-ancillas    : 9–12 (measure Z⊗…⊗Z stabilizers)
#   X-ancillas    : 13–16 (measure X⊗…⊗X stabilizers)

data_qubits = list(range(9))
z_ancillas  = [9, 10, 11, 12]
x_ancillas  = [13, 14, 15, 16]

# Stabilizer support: which data qubits each ancilla couples to
z_stabs = {9: [0,1], 10: [1,2,4,5], 11: [3,4,6,7], 12: [7,8]}
x_stabs = {13: [0,1,3,4], 14: [2,5], 15: [3,6], 16: [4,5,7,8]}

# ─────────────────────────────────────────────────────────────────────────────
# 3. Build the syndrome extraction circuit using Tsim
# ─────────────────────────────────────────────────────────────────────────────
# CNOT schedule (4 parallel steps) from the reference STAR stim circuit:
#   CX ancilla → data   for X stabilisers  (propagates X errors to ancilla)
#   CX data → ancilla   for Z stabilisers  (propagates Z errors to ancilla)

CNOT_STEPS = [
    "CX 13 0  14 2  16 4   1 10   3 11   7 12",
    "CX 13 3  14 5  16 7   2 10   4 11   8 12",
    "CX 13 1  15 3  16 5   0 9    4 10   6 11",
    "CX 13 4  15 6  16 8   1 9    5 10   7 11",
]

def syndrome_round(noise_p: float = 0.0) -> str:
    """Return stim program text for one complete syndrome extraction round.

    Produces 8 measurement bits per shot:
        bits 0-3  →  Z-ancilla outcomes  (q9, q10, q11, q12)
        bits 4-7  →  X-ancilla outcomes  (q13, q14, q15, q16)
    """
    lines = [
        "R  9 10 11 12",    # Z ancillas → |0⟩
        "RX 13 14 15 16",   # X ancillas → |+⟩
        "TICK",
    ]
    all_q = " ".join(str(q) for q in range(17))
    for step in CNOT_STEPS:
        lines.append(step)
        if noise_p > 0:
            lines.append(f"DEPOLARIZE1({noise_p}) {all_q}")
        lines.append("TICK")
    lines += [
        "M  9 10 11 12",    # Z-basis measurement → syndrome bits
        "MX 13 14 15 16",   # X-basis measurement → syndrome bits
    ]
    return "\n".join(lines)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Noiseless simulation: two syndrome rounds
# ─────────────────────────────────────────────────────────────────────────────
# Data qubits initialised in |0⟩:
#   • Z stabilisers are +1 eigenstates → all Z syndrome bits = 0
#   • X stabilisers are NOT +1 eigenstates of |0⟩ → first-round X outcome
#     is random, but the CHANGE between rounds must be 0 (no noise)

circuit_noiseless = tsim.Circuit(
    "R " + " ".join(str(q) for q in data_qubits) + "\n"
    "TICK\n"
    + syndrome_round() + "\n"
    "TICK\n"
    + syndrome_round()
)

show_circuit(circuit_noiseless)

print(f"\nNoiseless circuit: {circuit_noiseless.num_qubits} qubits, "
      f"{circuit_noiseless.num_measurements} measurements (8 per round × 2 rounds)")

sampler = circuit_noiseless.compile_sampler(seed=42)
shots = 8
raw = sampler.sample(shots).astype(int)

# Unpack: round 1 → cols 0-7, round 2 → cols 8-15
z_r1, x_r1 = raw[:, 0:4], raw[:, 4:8]
z_r2, x_r2 = raw[:, 8:12], raw[:, 12:16]

print("\n── Noiseless (8 shots) ──")
print("Round-1 Z syndromes [q9 q10 q11 q12]  (expect all 0):")
print(" ", z_r1)
print("Round-1 X syndromes [q13 q14 q15 q16]  (random – first projection):")
print(" ", x_r1)
print("Δ_Z = r2 ⊕ r1  (expect all 0):")
print(" ", z_r1 ^ z_r2)
print("Δ_X = r2 ⊕ r1  (expect all 0):")
print(" ", x_r1 ^ x_r2)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Deterministic error: X flip on qubit 4 between the two rounds
# ─────────────────────────────────────────────────────────────────────────────
# X error on q4 anticommutes with every Z stabiliser that contains q4:
#   Z_10 = Z1·Z2·Z4·Z5  → fires  (Δ = 1)
#   Z_11 = Z3·Z4·Z6·Z7  → fires  (Δ = 1)
#   Z_9, Z_12           → silent
#   All X stabilisers commute with X error → Δ_X = 0

circuit_xerr = tsim.Circuit(
    "R " + " ".join(str(q) for q in data_qubits) + "\n"
    "TICK\n"
    + syndrome_round() + "\n"
    "TICK\n"
    "X_ERROR(1.0) 4\n"
    "TICK\n"
    + syndrome_round()
)

s_err = circuit_xerr.compile_sampler(seed=0).sample(4).astype(int)
dz = s_err[:, 8:12] ^ s_err[:, 0:4]
dx = s_err[:, 12:16] ^ s_err[:, 4:8]

print("\n── Deterministic X error on qubit 4 ──")
print("Δ_Z [q9, q10, q11, q12]  (expect [0,1,1,0]):")
print(" ", dz)
print("Δ_X [q13, q14, q15, q16]  (expect [0,0,0,0]):")
print(" ", dx)

# ─────────────────────────────────────────────────────────────────────────────
# 6. Noisy simulation: DEPOLARIZE1(p), measure syndrome change rates
# ─────────────────────────────────────────────────────────────────────────────

p = 0.01
circuit_noisy = tsim.Circuit(
    "R " + " ".join(str(q) for q in data_qubits) + "\n"
    "TICK\n"
    + syndrome_round(noise_p=p) + "\n"
    "TICK\n"
    + syndrome_round(noise_p=p)
)

n_shots = 2000
s_noisy = circuit_noisy.compile_sampler(seed=7).sample(n_shots).astype(int)
dz_noisy = s_noisy[:, 8:12] ^ s_noisy[:, 0:4]
dx_noisy = s_noisy[:, 12:16] ^ s_noisy[:, 4:8]

print(f"\n── Depolarising noise p = {p}, N = {n_shots} shots ──")
print("Z-ancilla syndrome change rates  (heavier stabilisers fire more often):")
for i, (anc, label) in enumerate({9:"Z_9", 10:"Z_10", 11:"Z_11", 12:"Z_12"}.items()):
    rate = dz_noisy[:, i].mean()
    print(f"  {label} (data {z_stabs[anc]}, weight {len(z_stabs[anc])}): {rate:.3f}")
print("X-ancilla syndrome change rates:")
for i, (anc, label) in enumerate({13:"X_13", 14:"X_14", 15:"X_15", 16:"X_16"}.items()):
    rate = dx_noisy[:, i].mean()
    print(f"  {label} (data {x_stabs[anc]}, weight {len(x_stabs[anc])}): {rate:.3f}")


TypeError: unhashable type: 'Circuit'

## Team STAR Part 2: Estimate STAR Fidelities
Review the STAR circuits that have been provided in the assets folder and use Tsim to simulate them in a noisy setting. 
Reproduce the fidelity plot below using data from your Tsim simulations.

![star_sim.svg](./assets/star_sim.svg)

The provided circuits are for a default rotation angle of 0.01*pi. To simulate different rotation angles, you can use the following function to compute the physical rotation angle needed to achieve a logical rotation of angle `logical_angle_in_pi` on `num_physical_rotations` physical rotations.

Also make sure to check out the comments in `circuits/star_d=3.stim` to get some hints about the circuit structure.
```python
def physical_angle(logical_angle_in_pi: float, num_physical_rotations: int) -> float:
    """
    Compute the physical rotation angle needed to achieve a logical rotation of
    angle `logical_angle_in_pi` on `num_physical_rotations` physical rotations.

    Args:
        logical_angle_in_pi (float): The logical rotation angle in units of pi.
        num_physical_rotations (int): The number of physical rotations that are applied.
    Returns:
        float: The physical rotation angle in units of pi.
    """

    assert (
        num_physical_rotations % 2 == 1 and num_physical_rotations > 0
    ), "k must be a positive odd integer"
    sign = -1 if (num_physical_rotations + 1) % 4 == 0 else 1
    logical_angle_in_rad = logical_angle_in_pi * np.pi
    x = np.tan(logical_angle_in_rad / 2) ** (1 / num_physical_rotations)
    theta_phys = 2 * np.arctan(x)
    return float(sign * theta_phys / np.pi)
```

**Goal:** Learn how to load and run pre-built circuits using Tsim. 


In [ ]:
# part 2

## Team STAR Part 3 — Teleport a non-Clifford rotation into a logical qubit
Now assume a noiseless setting but where non-Clifford gates cannot be applied directly to the main logical qubit. 
Construct a protocol that uses an ancillary logical qubit to teleport a small-angle rotation into the main qubit while assuming the STAR transversal architecture.
Important: you will need to use postselection to filter results because Tsim (unlike PyQrack) does not support feed-forwarded operations.

![rus.png](./assets/rus.png)

Figure taken from *"Partially Fault-Tolerant Quantum Computing Architecture with Error-Corrected Clifford Gates and Space-Time Efficient Analog Rotations"* by *Akahoshi et al.*

**Goal:** Show how STAR enables indirect implementation of small-angle non-Clifford logical rotations, and analyze the costs of doing so. 


In [ ]:
# part 3